# Meetings - Notes & Questions

## Modeling interest rates

notes ...

## Deep hedging

Fait:
- Deep hedging vs Black-Scholes avec et sans transaction costs


En cours:
- Objectif de faire comme dans le papier sur RL hedging options, mais juste avec le RL et des simulations differente (pas encore d'option), faire les memes graphique et analyser
- Simuler Heston, GARCH, Jump-diffusion process, afin de voir la difference entre deep hedging et black-scholes hedging
- Train deep hedging agent sur plusieurs model (batch-style)

Question:
- utiliser Black-Scholes delta comme base sur d'autre model (ex. Heston)
- Overview des references vers Francois et Al. pour simuler des options prices
- Comment fonctionne delta-gamma hedging (hedge avec un option supplementaire, transaction costs?)


a apprendre:

Autoencoder
NN generatif


## SANOS: Smooth strictly Arbitrage-free Non-parametric Option Surfaces

Summary of Buehler, Horvath, Kratsios, Limmer, Saqur (2026) - `SANOS - Option surface.pdf`

**Core idea**: A method to build option price surfaces $\hat C(T,K)$ that are simultaneously *smooth* and *strictly arbitrage-free* - something existing approaches (SSVI, plain linear interpolation, heavy stochastic-vol calibration) don't achieve together. It's framed as a smooth generalization of the well-known linear interpolation scheme for arbitrage-free option prices.

**Key construction** (eq. 1):
$$\hat C(T_j, K) = \sum_i q_j^i \,\text{Call}(K_j^i, K, V_j)$$
Call prices are convex combinations of Black-Scholes call payoffs anchored at model strikes $K_j^i$, with weights $q_j^i$ that behave like a discrete martingale transition density (must be non-negative, sum to 1, and satisfy the martingale property $K_j \cdot q_j = K_{j-1} \cdot q_{j-1}$-style constraints). Setting the smoothness parameter $\eta=0$ recovers plain linear interpolation (non-smooth); $\eta \to 1$ over-smooths to almost only fitting ATM. $\eta=0.25$ is their recommended default.

**Why it matters**: Linear interpolation is arbitrage-free but is proven (citing Buehler 2006) to be "the most expensive" interpolation - it systematically overprices between quoted strikes, which shows up as implied vol bumps between strikes (Figure 2). SANOS fixes that while keeping strict no-arbitrage.

**Theoretical backbone**: Theorem 2.2 gives 5 shape conditions (unit expectation, no atom at 0, decay to 0, convexity in K, monotonicity in T) that are necessary and sufficient for a call surface to correspond to an actual positive martingale (Fundamental Theorem of Asset Pricing). Theorem 3.1 shows that replacing the discrete jump anchor with call prices under a smooth martingale $Y$ (they use log-normal, i.e. Black-Scholes) preserves all 5 conditions, hence "smooth + arbitrage-free."

**Calibration**: Fitting $q$ to market bid/ask is a **linear program** (or convex program with bid/ask penalties) - this is the big practical win: no nonlinear optimization, sub-second fits to full SPX surfaces across 48 expiries (91.4% of options fit within bid/ask).

**Extra contribution**: An equivalent parametrization via "discrete local volatilities" (Sigma, extending Buehler & Ryskin 2015) where the *only* constraint is positivity - useful for generative/ML models of option surfaces, since you can just exponentiate or square unconstrained NN outputs to get valid parameters.

**Relevant to project**: Could feed directly into building a smooth vol surface as an input/testbed for Deep Hedging plots (see `Ploting_DH.py`).